In [1]:
from InSitu14CO import Propagator
import numpy as np
import matplotlib.pyplot as plt
import crflux.models as pm
import matplotlib.ticker as ticker
import matplotlib as mpl
import Functions_14CO as F

from tqdm import tqdm

import pandas as pd

from scipy import stats

import proposal as pp

In [2]:
# setup pyplot formatting

axes_style = { 'grid'      : 'True',
               'labelsize' : '14',
               'labelpad'  : '8.0'
             }
grid_style = { 'alpha'     : '0.75',
               'linestyle' : ':' }
font_style = { 'size'      : '10' }

mpl.rc('font', **font_style)
mpl.rc('axes', **axes_style)
mpl.rc('grid', **grid_style)

In [3]:
Prop = Propagator(logE_mu_max=7.5)

In [4]:
def get_proposal(Prop, mu_pos=True):
    if mu_pos:
        mu = pp.particle.MuPlusDef()
    else:
        mu = pp.particle.MuMinusDef()
    cuts = pp.EnergyCutSettings(500, 0.05, True)

    medium = pp.medium.Water()

    args = {"particle_def": mu, "target": medium, "interpolate": True, "cuts": cuts}

    # Initialise standard cross-sections, then specify and set parametrisation models

    cross_sections = pp.crosssection.make_std_crosssection(**args)

    brems_param = pp.parametrization.bremsstrahlung.KelnerKokoulinPetrukhin(lpm=False)
    epair_param = pp.parametrization.pairproduction.KelnerKokoulinPetrukhin(lpm=False)
    ionis_param = pp.parametrization.ionization.BetheBlochRossi(energy_cuts=cuts)
    shado_param = pp.parametrization.photonuclear.ShadowButkevichMikheyev()
    photo_param = pp.parametrization.photonuclear.AbramowiczLevinLevyMaor97(
        shadow_effect=shado_param
    )

    cross_sections[0] = pp.crosssection.make_crosssection(brems_param, **args)
    cross_sections[1] = pp.crosssection.make_crosssection(epair_param, **args)
    cross_sections[2] = pp.crosssection.make_crosssection(ionis_param, **args)
    cross_sections[3] = pp.crosssection.make_crosssection(photo_param, **args)

    # Propagation utility

    collection = pp.PropagationUtilityCollection()

    collection.interaction = pp.make_interaction(cross_sections, True)
    collection.displacement = pp.make_displacement(cross_sections, True)
    collection.time = pp.make_time(cross_sections, mu, True)
    collection.decay = pp.make_decay(cross_sections, mu, True)

    pp.PropagationUtilityCollection.cont_rand = False

    utility = pp.PropagationUtility(collection=collection)

    # Other settings

    pp.do_exact_time = False

    # Set up geometry

    detector = pp.geometry.Sphere(
        position=pp.Cartesian3D(0, 0, 0), radius=10000000, inner_radius=0
    )
    density_distr = pp.density_distribution.density_homogeneous(
        mass_density=Prop.rho_ice
    )

    return pp.Propagator(mu, [(detector, utility, density_distr)])

In [5]:
def proposal_loop(Prop, propagator, N, energy):
    
    mu_initial = pp.particle.ParticleState()
    mu_initial.energy = (energy + Prop.mu_mass) * 1e3 # Muon Total Energy (MeV)
    mu_initial.position = pp.Cartesian3D(0, 0, 0)
    mu_initial.direction = pp.Cartesian3D(0, 0, -1)

    slant_depth = Prop.h_bins[-1]/Prop.cosTH[-1]/Prop.rho_ice * 1e2 # convert meters-water-equivalent to cm
    
    print ('Running {} Simulations at {:.1e} GeV...'.format(N, energy))

    tracks = [propagator.propagate(mu_initial, slant_depth) for i in tqdm(range(N))]
    
    E = np.concatenate([np.array(t.track_energies()) * 1e-3 - Prop.mu_mass for t in tracks]) # convert MeV Total to GeV Kinetic
    D = np.concatenate([np.array(t.track_propagated_distances()) * Prop.rho_ice * 1e-2 for t in tracks]) # convert cm to m.w.e slant depth
    
    counts = np.zeros((len(Prop.cosTH), len(Prop.E_mu), len(Prop.h_bins)), dtype=int)
    
    i_depths = np.digitize(D*Prop.cosTH.reshape((-1,1)), Prop.h_bins, right=True)
    i_energies = np.digitize(E, Prop.E_mu_bins[:-1])-1
    
    # Count muons into energy bins for each depth and zenith angle
    # This method takes advantage of the fact that each track starts at 0 distance traveled
    # Thus, if the next recorded event occurred at 0 distance, it's from the next track, and so the current one is the last event of this track

    # Loop over each event recorded
    # We start at -1 because it makes indexing easier
    for i in tqdm(range(-1, len(D)-1)):
        if i_energies[i] != -1:
            for j,d in enumerate(i_depths):
                if d[i]<d[i+1]:
                    counts[j, i_energies[i], d[i]:d[i+1] if D[i+1]!=0. else d[i]:] += 1
                    
    return counts/N

In [6]:
def get_survival_tensor(Prop, N=10**5):
    survival_tensor = np.zeros((2,len(Prop.E_mu),len(Prop.cosTH),len(Prop.E_mu),len(Prop.h_bins)))
    
    for i,mu_pos in enumerate([True, False]):
        propagator = get_proposal(Prop, mu_pos)
        for j,energy in enumerate(Prop.E_mu[:40]):
            survival_tensor[i,j] = proposal_loop(Prop, propagator, N, energy)
    
    return survival_tensor

In [13]:
survival_tensor = get_survival_tensor(Prop)

Running 100000 Simulations at 1.1e-01 GeV...


100%|███████████████████████████████████████████████████████████████████████| 100000/100000 [00:00<00:00, 381292.74it/s]


Simulations Finished.
Constructing Survival Tensor...


100%|███████████████████████████████████████████████████████████████████████| 300000/300000 [00:01<00:00, 165359.69it/s]


Survival Tensor Finished.

Running 100000 Simulations at 1.4e-01 GeV...


...

Survival Tensor Finished.

Running 100000 Simulations at 8.9e+02 GeV...


100%|█████████████████████████████████████████████████████████████████████████| 100000/100000 [00:47<00:00, 2084.13it/s]


Simulations Finished.
Constructing Survival Tensor...


100%|███████████████████████████████████████████████████████████████████| 17506294/17506294 [01:59<00:00, 146090.06it/s]


Survival Tensor Finished.



In [24]:
np.save('survival_tensor_TEST',st)